In [11]:
import pandas as pd
df =pd.read_csv("../data/milestone1_ayesha-naaz.csv")
df.head()

,id,sender,subject,body,priority,triage_label,clean_text,triage
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,reminder the client meeting is scheduled at t...,respond_or_act
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,your invoice of inr is due on please pay to ...,respond_or_act
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,reminder the client meeting is scheduled at t...,respond_or_act
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,hello team please find the attached weekly rep...,respond_or_act
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,hello team please find the attached weekly rep...,respond_or_act


In [12]:
df.shape


(200, 8)

In [8]:
# import pandas as pd
# df =pd.read_csv("../data/sample_emails_with_triage_200.csv")
# df.head()


import os

print(os.path.abspath("../data/sample_emails_with_triage_200.csv"))


c:\Users\skgha\OneDrive\Desktop\Email Assisstant\infosys-langgraph-email-assistant-group2\data\sample_emails_with_triage_200.csv


In [13]:
def triage_rule(text):
    t = str(text)

    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return 'notify_human'

    if any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return 'ignore'

    if any(k in t for k in ['invoice','payment','overdue','due on','meeting']):
        return 'respond_or_act'

    return 'respond_or_act'


In [14]:
df['triage'] = df['clean_text'].apply(triage_rule)
df[['clean_text','triage']].head()

#The triage function is applied to the cleaned text

,clean_text,triage
0,reminder the client meeting is scheduled at t...,respond_or_act
1,your invoice of inr is due on please pay to ...,respond_or_act
2,reminder the client meeting is scheduled at t...,respond_or_act
3,hello team please find the attached weekly rep...,respond_or_act
4,hello team please find the attached weekly rep...,respond_or_act


In [7]:
df.columns


Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label'], dtype='object')

In [15]:
eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label,clean_text,triage
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore,your order has been shipped and is expected t...,respond_or_act
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human,hi dont miss our sale with discounts up to on...,ignore
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human,notice your account will be locked unless veri...,respond_or_act
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond,reminder the client meeting is scheduled at t...,respond_or_act
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond,your order has been shipped and is expected t...,respond_or_act


In [16]:
#create a column of ideal_response(if elif logic)

def ideal_response(text):
    """
    Returns the ideal response based on email content.
    """
    t = str(text).lower()  # ensure lowercase for matching

    # Security-related emails
    if any(k in t for k in ['password', 'reset password', 'security issue']):
        return "Please escalate this security issue to the IT team immediately."

    # Marketing / promotion emails
    elif any(k in t for k in ['promotion', 'sale', 'offer', 'unsubscribe', 'newsletter']):
        return "No action needed. This is a promotional email."

    # Payment / invoice / meeting emails
    elif any(k in t for k in ['invoice', 'payment', 'overdue', 'due on', 'meeting']):
        return "Please respond appropriately to the client regarding payment or schedule."

    # Default response
    else:
        return "Please read the email and respond as necessary."


In [17]:
df['ideal_response'] = df['clean_text'].apply(ideal_response)
df[['clean_text', 'triage', 'ideal_response']].head()


,clean_text,triage,ideal_response
0,reminder the client meeting is scheduled at t...,respond_or_act,Please respond appropriately to the client reg...
1,your invoice of inr is due on please pay to ...,respond_or_act,Please respond appropriately to the client reg...
2,reminder the client meeting is scheduled at t...,respond_or_act,Please respond appropriately to the client reg...
3,hello team please find the attached weekly rep...,respond_or_act,Please read the email and respond as necessary.
4,hello team please find the attached weekly rep...,respond_or_act,Please read the email and respond as necessary.


In [ ]:
Create a new CSV file, naming it as evaluation for scoring emails with minimum 100 mails. 
Load the file and run your existing agent from milestone1 and generate outputs and save in another CSV file, and create a simple rule-based scoring like tone score, accent score, clarity score.

In [1]:
import pandas as pd
import re


In [4]:
sample_data = pd.read_csv("../data/email_evaluation_dataset_ayesha.csv")
sample_data.head()


,email_text,expected_action,expected_tone
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,urgent
2,We are pleased to inform you that your interns...,respond,polite
3,Happy New Year! Wishing you success and good h...,ignore,neutral
4,Please find attached the minutes of yesterday’...,ignore,neutral


In [5]:
sample_data['clean_text'] = (
    sample_data['email_text']
    .astype(str)
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

sample_data[['email_text', 'clean_text']].head()


,email_text,clean_text
0,Reminder: Client meeting scheduled for Jan 10 ...,reminder client meeting scheduled for jan at ...
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,your invoice inv is due on jan kindly make th...
2,We are pleased to inform you that your interns...,we are pleased to inform you that your interns...
3,Happy New Year! Wishing you success and good h...,happy new year wishing you success and good he...
4,Please find attached the minutes of yesterday’...,please find attached the minutes of yesterdays...


In [6]:
def triage_rule(text):
    t = str(text)

    if any(k in t for k in ['password', 'reset', 'security', 'login']):
        return 'notify_human'

    if any(k in t for k in ['promotion', 'sale', 'offer', 'newsletter']):
        return 'ignore'

    if any(k in t for k in ['invoice', 'payment', 'meeting', 'due', 'submit']):
        return 'respond_or_act'

    return 'respond_or_act'


In [7]:
sample_data['predicted_action'] = sample_data['clean_text'].apply(triage_rule)

sample_data[['email_text', 'expected_action', 'predicted_action']].head()


,email_text,expected_action,predicted_action
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,respond_or_act
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,respond_or_act
2,We are pleased to inform you that your interns...,respond,respond_or_act
3,Happy New Year! Wishing you success and good h...,ignore,respond_or_act
4,Please find attached the minutes of yesterday’...,ignore,respond_or_act


In [8]:
def tone_score(text):
    if any(w in text for w in ['urgent', 'immediately', 'asap']):
        return 5
    if any(w in text for w in ['please', 'kindly', 'thank']):
        return 4
    return 3


In [9]:
import re

def accent_score(text):
    if re.search(r'[^a-zA-Z0-9 .,]', text):
        return 3
    return 5


In [10]:
def clarity_score(text):
    words = len(text.split())
    if words < 5:
        return 2
    if words > 40:
        return 3
    return 5


In [11]:
sample_data['tone_score'] = sample_data['clean_text'].apply(tone_score)
sample_data['accent_score'] = sample_data['clean_text'].apply(accent_score)
sample_data['clarity_score'] = sample_data['clean_text'].apply(clarity_score)

sample_data.head()


,email_text,expected_action,expected_tone,clean_text,predicted_action,tone_score,accent_score,clarity_score
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite,reminder client meeting scheduled for jan at ...,respond_or_act,4,5,5
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,urgent,your invoice inv is due on jan kindly make th...,respond_or_act,4,5,5
2,We are pleased to inform you that your interns...,respond,polite,we are pleased to inform you that your interns...,respond_or_act,4,5,5
3,Happy New Year! Wishing you success and good h...,ignore,neutral,happy new year wishing you success and good he...,respond_or_act,3,5,5
4,Please find attached the minutes of yesterday’...,ignore,neutral,please find attached the minutes of yesterdays...,respond_or_act,4,5,5


# Milestone 2 Assignment : Email Assistant Evaluation

## Objective: 
Apply the email assistant to classify emails by action and tone, evaluate accuracy, perform error analysis, and reflect on results.

## Tasks: 
Load & clean data, define rules for action & tone, generate predictions, compare with expected values, calculate accuracy, analyze errors, save output, and reflect.


In [37]:
# ==============================
# Step 1: Load Dataset and Clean Email Text
# ==============================

import pandas as pd
import re

# Load the email dataset
df = pd.read_csv("../data/email_evaluation_dataset_ayesha.csv")

# Clean the email text:
# - convert to lowercase
# - remove special characters except letters and numbers
df['clean_text'] = (
    df['email_text']
    .astype(str)
    .str.lower()
    .str.replace('[^a-zA-Z0-9 ]', '', regex=True)
)

# Preview the cleaned dataset
df[['email_text', 'clean_text']].head()


,email_text,clean_text
0,Reminder: Client meeting scheduled for Jan 10 ...,reminder client meeting scheduled for jan 10 a...
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,your invoice inv1023 is due on jan 5 kindly ma...
2,We are pleased to inform you that your interns...,we are pleased to inform you that your interns...
3,Happy New Year! Wishing you success and good h...,happy new year wishing you success and good he...
4,Please find attached the minutes of yesterday’...,please find attached the minutes of yesterdays...


In [38]:
# ==============================
# Step 2: Improved Action Detection
# ==============================

def improved_action_v3(text):
    """
    Returns predicted action with expanded keywords for higher accuracy.
    Actions:
    - 'notify_human' : security-related issues
    - 'ignore'       : promotions, greetings, newsletters
    - 'respond'      : payment, invoice, meetings, reminders
    """
    t = str(text)
    
    # Security-related emails
    if any(k in t for k in ['password', 'reset', 'security', 'login', 'account locked', 'login issue']):
        return 'notify_human'
    
    # Marketing / promotion / greetings
    if any(k in t for k in ['promotion', 'sale', 'offer', 'newsletter', 
                            'happy new year', 'congratulations', 'greetings']):
        return 'ignore'
    
    # Payment / invoice / meeting / reminders / follow-ups
    if any(k in t for k in ['invoice', 'payment', 'due', 'submit', 
                            'reminder', 'meeting scheduled', 'follow up', 'overdue']):
        return 'respond'
    
    # Default action
    return 'respond'

# Test the improved function on first 5 emails
df['predicted_action_test'] = df['clean_text'].apply(improved_action_v3)
df[['email_text', 'predicted_action_test']].head()


,email_text,predicted_action_test
0,Reminder: Client meeting scheduled for Jan 10 ...,respond
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond
2,We are pleased to inform you that your interns...,respond
3,Happy New Year! Wishing you success and good h...,ignore
4,Please find attached the minutes of yesterday’...,respond


In [39]:
# ==============================
# Step 3: Improved Tone Detection & Combined Assistant
# ==============================

def improved_tone(text):
    """
    Returns predicted tone for an email.
    Tones:
    - 'urgent' : indicates immediate attention
    - 'polite' : uses courteous words
    - 'neutral': default tone
    """
    t = str(text)
    
    # Urgent cues
    if any(k in t for k in ['urgent', 'immediately', 'asap', 'due today', 'deadline', 'action required']):
        return 'urgent'
    
    # Polite cues
    if any(k in t for k in ['please', 'kindly', 'thank you', 'appreciate']):
        return 'polite'
    
    # Default tone
    return 'neutral'

def email_assistant_v3(text):
    """
    Returns both predicted action and predicted tone for an email.
    """
    action = improved_action_v3(text)
    tone = improved_tone(text)
    return action, tone

# Apply the assistant function to each email
df['assistant_output'] = df['clean_text'].apply(email_assistant_v3)

# Split the tuple into separate columns
df[['predicted_action', 'predicted_tone']] = pd.DataFrame(
    df['assistant_output'].tolist(), index=df.index
)

# Drop the combined column
df.drop(columns=['assistant_output'], inplace=True)

# Preview predictions
df[['email_text', 'predicted_action', 'predicted_tone']].head()


,email_text,predicted_action,predicted_tone
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,polite
2,We are pleased to inform you that your interns...,respond,polite
3,Happy New Year! Wishing you success and good h...,ignore,neutral
4,Please find attached the minutes of yesterday’...,respond,polite


In [40]:
# ==============================
# Step 4: Compare Predictions with Expected Values
# ==============================

# Compare predicted action with expected action
df['action_correct'] = df['predicted_action'] == df['expected_action']

# Compare predicted tone with expected tone
df['tone_correct'] = df['predicted_tone'] == df['expected_tone']

# Preview results
df[['email_text', 'predicted_action', 'expected_action', 'action_correct',
    'predicted_tone', 'expected_tone', 'tone_correct']].head()


,email_text,predicted_action,expected_action,action_correct,predicted_tone,expected_tone,tone_correct
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,respond,True,polite,polite,True
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,respond,True,polite,urgent,False
2,We are pleased to inform you that your interns...,respond,respond,True,polite,polite,True
3,Happy New Year! Wishing you success and good h...,ignore,ignore,True,neutral,neutral,True
4,Please find attached the minutes of yesterday’...,respond,ignore,False,polite,neutral,False


In [41]:
# ==============================
# Step 5: Calculate Accuracy
# ==============================

# Calculate Action Accuracy (%)
action_accuracy = df['action_correct'].mean() * 100

# Calculate Tone Accuracy (%)
tone_accuracy = df['tone_correct'].mean() * 100

# Print accuracies
print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")

# Optional: create a summary DataFrame
accuracy_summary = pd.DataFrame({
    'Metric': ['Action Accuracy', 'Tone Accuracy'],
    'Accuracy (%)': [action_accuracy, tone_accuracy]
})
accuracy_summary


Action Accuracy: 41.00%
Tone Accuracy: 61.00%


,Metric,Accuracy (%)
0,Action Accuracy,41.0
1,Tone Accuracy,61.0


In [42]:
# ==============================
# Step 6: Perform Error Analysis
# ==============================

# Identify emails where action prediction failed
action_errors = df[df['action_correct'] == False][
    ['email_text', 'expected_action', 'predicted_action']
]

# Identify emails where tone prediction failed
tone_errors = df[df['tone_correct'] == False][
    ['email_text', 'expected_tone', 'predicted_tone']
]

# Identify emails where either action or tone failed
all_errors = df[(df['action_correct'] == False) | (df['tone_correct'] == False)][
    ['email_text', 'expected_action', 'predicted_action',
     'expected_tone', 'predicted_tone']
]

# Preview action errors
print("Action Errors:")
display(action_errors.head(10))

# Preview tone errors
print("Tone Errors:")
display(tone_errors.head(10))

# Preview all errors
print("All Errors:")
display(all_errors.head(10))


Action Errors:


,email_text,expected_action,predicted_action
4,Please find attached the minutes of yesterday’...,ignore,respond
5,Reminder: Submit your assignment by tonight 11...,notify,respond
7,Your payment has been successfully received. T...,ignore,respond
8,We noticed unusual login activity on your acco...,notify,notify_human
10,The office will remain closed on Monday due to...,notify,respond
12,Your internship completion certificate is now ...,notify,respond
13,Reminder: Library books are due for return tom...,notify,respond
14,Thank you for attending the workshop. We value...,ignore,respond
16,Your exam hall ticket has been released.,notify,respond
18,Payment failed due to insufficient balance. Pl...,notify,respond


Tone Errors:


,email_text,expected_tone,predicted_tone
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,urgent,polite
4,Please find attached the minutes of yesterday’...,neutral,polite
5,Reminder: Submit your assignment by tonight 11...,urgent,neutral
6,Can you share the updated project timeline by ...,polite,neutral
7,Your payment has been successfully received. T...,neutral,polite
8,We noticed unusual login activity on your acco...,urgent,neutral
9,Invitation to attend the alumni networking eve...,polite,neutral
13,Reminder: Library books are due for return tom...,urgent,neutral
14,Thank you for attending the workshop. We value...,neutral,polite
15,Can we reschedule our call to next week?,polite,neutral


All Errors:


,email_text,expected_action,predicted_action,expected_tone,predicted_tone
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,respond,urgent,polite
4,Please find attached the minutes of yesterday’...,ignore,respond,neutral,polite
5,Reminder: Submit your assignment by tonight 11...,notify,respond,urgent,neutral
6,Can you share the updated project timeline by ...,respond,respond,polite,neutral
7,Your payment has been successfully received. T...,ignore,respond,neutral,polite
8,We noticed unusual login activity on your acco...,notify,notify_human,urgent,neutral
9,Invitation to attend the alumni networking eve...,respond,respond,polite,neutral
10,The office will remain closed on Monday due to...,notify,respond,neutral,neutral
12,Your internship completion certificate is now ...,notify,respond,neutral,neutral
13,Reminder: Library books are due for return tom...,notify,respond,urgent,neutral


In [45]:
# ==============================
# Step 7: Save the Final Output CSV
# ==============================

# Define output path and file name
output_path = "../data/milestone2_output_evaluation.csv"

# Save the DataFrame with predictions and comparison columns
df.to_csv(output_path, index=False)

print(f"Final CSV saved to: {output_path}")


Final CSV saved to: ../data/milestone2_output_evaluation.csv


In [43]:
df.columns


Index(['email_text', 'expected_action', 'expected_tone', 'clean_text',
       'predicted_action_test', 'predicted_action', 'predicted_tone',
       'action_correct', 'tone_correct'],
      dtype='object')

In [44]:
df.head(5)


,email_text,expected_action,expected_tone,clean_text,predicted_action_test,predicted_action,predicted_tone,action_correct,tone_correct
0,Reminder: Client meeting scheduled for Jan 10 ...,respond,polite,reminder client meeting scheduled for jan 10 a...,respond,respond,polite,True,True
1,Your invoice INV-1023 is due on Jan 5. Kindly ...,respond,urgent,your invoice inv1023 is due on jan 5 kindly ma...,respond,respond,polite,True,False
2,We are pleased to inform you that your interns...,respond,polite,we are pleased to inform you that your interns...,respond,respond,polite,True,True
3,Happy New Year! Wishing you success and good h...,ignore,neutral,happy new year wishing you success and good he...,ignore,ignore,neutral,True,True
4,Please find attached the minutes of yesterday’...,ignore,neutral,please find attached the minutes of yesterdays...,respond,respond,polite,False,False


## Which type of emails were hardest to classify?
Emails with mixed intent (e.g., reminders that are also polite or security notifications disguised as routine emails).

Emails without explicit keywords for action or tone (e.g., “minutes of yesterday’s meeting” or “library book reminder”).

## Why did your rules fail in some cases?
Rule-based systems rely entirely on explicit keywords.

Subtle phrasing, missing keywords, or context-dependent meaning caused misclassification.

Tone is especially hard to capture because politeness or urgency can be implied rather than stated.

## How could an LLM improve this process?
LLMs can understand context and implied meaning, not just keywords.

They can classify nuanced emails accurately (e.g., polite reminders vs. urgent deadlines).

LLMs can also adapt to new phrasing without manually updating rules.